In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("snap/amazon-fine-food-reviews")

print("Path to dataset files:", path)

In [ ]:
import pandas as pd
import os

# Assuming the 'path' variable from the previous cell is available and contains the dataset directory.
# A common file in the 'amazon-fine-food-reviews' dataset is 'Reviews.csv'.
df = pd.read_csv(os.path.join(path, 'Reviews.csv'))

df.isnull().sum()

In [ ]:
df.columns

# Convert Ratings to Sentiment Labels

In [ ]:
def sentiment_label(score):
    if score >= 4:
        return "positive"
    elif score == 3:
        return "neutral"
    else:
        return "negative"

df['Sentiment'] = df['Score'].apply(sentiment_label)

df[['Score', 'Sentiment']].head()

In [ ]:
df = df[['Text', 'Sentiment']]

df.head()

# Check Class Distribution

In [ ]:
df['Sentiment'].value_counts()

# Reduce Dataset Size

In [ ]:
df = df.sample(20000, random_state=42)

In [ ]:
df = df.reset_index(drop=True)

# Basic Text Cleaning

In [ ]:
!pip install nltk

In [ ]:
import re
import nltk
nltk.download('stopwords')

from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    words = text.split()
    words = [word for word in words if word not in stop_words]

    return " ".join(words)

In [ ]:
df['Cleaned_Text'] = df['Text'].apply(clean_text)

df.head()

# Encode Labels

In [ ]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

df['label'] = encoder.fit_transform(df['Sentiment'])

df.head()

# Train Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df['Cleaned_Text'],
    df['label'],
    test_size=0.2,
    random_state=42
)

In [ ]:
train_data = pd.DataFrame({
    'text': X_train,
    'label': y_train
})

test_data = pd.DataFrame({
    'text': X_test,
    'label': y_test
})

train_data.to_csv("train_data.csv", index=False)
test_data.to_csv("test_data.csv", index=False)

In [ ]:
from google.colab import files

files.download("train_data.csv")
files.download("test_data.csv")

MODEL TRAINING


In [ ]:
import warnings
warnings.filterwarnings("ignore")

from transformers import logging
logging.set_verbosity_error()

# ── 1. BERT ──────────────────────────────────────────────────────────────────
from transformers import BertTokenizer, BertForSequenceClassification
import torch

bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=3,
    ignore_mismatched_sizes=True
)

def bert_tokenize(texts):
    return bert_tokenizer(texts, padding=True, truncation=True, max_length=512, return_tensors='pt')

bert_optimizer = torch.optim.AdamW(bert_model.parameters(), lr=2e-5)

def bert_train(dataloader, epochs=3):
    bert_model.train()
    for epoch in range(epochs):
        for batch in dataloader:
            bert_optimizer.zero_grad()
            outputs = bert_model(input_ids=batch['input_ids'],
                                 attention_mask=batch['attention_mask'],
                                 labels=batch['labels'])
            outputs.loss.backward()
            bert_optimizer.step()
